# Qwen GRPO P100 probe

Profiles 20 GRPO steps on one P100: GPU architecture, prompt tokens, rollout/reward/policy/reference/save timings, low-variance groups, and peak memory. The run is capped at 10 minutes and saves checkpoints locally only.


In [ ]:
import os, subprocess, time
session_started = time.monotonic()
repo = '/kaggle/working/minesweeper-grpo'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', '--branch', 'optimize-kaggle-p100', '--single-branch', 'https://github.com/Anchal-T/minesweeper-grpo.git', repo], check=True)
print(f'[SETUP] repository={repo}')


In [ ]:
import os, subprocess, sys, threading, time
os.chdir('/kaggle/working/minesweeper-grpo')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'transformers', 'huggingface_hub'], check=True, timeout=120)
command = [sys.executable, 'grpo.py', '--init-from', 'base', '--steps', '20', '--prompts-per-step', '16', '--group', '8', '--micro-batch', '32', '--kl-coef', '0.05', '--adv-norm', 'none', '--profile-phases', '--verify-kv-cache', '--eval-every', '20', '--time-budget-min', '7.5', '--out-dir', 'runs/p100-probe', '--hub-repo', '']
print('[PROBE] launching one-GPU, 20-step GRPO timing run', flush=True)
proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def stream_output():
    for line in proc.stdout:
        print(line, end="", flush=True)
reader = threading.Thread(target=stream_output, daemon=True)
reader.start()
try:
    return_code = proc.wait(timeout=max(1, 570 - (time.monotonic() - session_started)))
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    reader.join()
    raise RuntimeError('P100 probe exceeded the 10-minute session cap')
reader.join()
if return_code:
    raise subprocess.CalledProcessError(return_code, proc.args)
print(f'[PROBE] complete elapsed_seconds={time.monotonic() - session_started:.1f}', flush=True)
